# 00 — Exploration du dataset

Objectifs (semaine 1) :
- Vérifier la structure `data/clean` / `data/noisy`
- Compter le nombre de paires alignées
- Distribution des durées des clips
- Écouter / regarder quelques exemples

Prérequis : avoir exécuté le setup de `main.ipynb` (mount Drive + clone repo + ajout au `sys.path`).

## Setup (à exécuter si on n'est pas passé par `main.ipynb` avant)

In [ ]:
import os, sys
REPO_DIR = '/content/Filtre-Voix-DL'
if os.path.exists(REPO_DIR) and REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

try:
    from IPython import get_ipython
    ipy = get_ipython()
    if ipy is not None:
        ipy.run_line_magic('load_ext', 'autoreload')
        ipy.run_line_magic('autoreload', '2')
except Exception as e:
    print(f'Autoreload non chargé ({e}) — sans impact.')

In [ ]:
import os
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt
import soundfile as sf

from src import config
from src.dataset import list_pairs
from src import audio as A

## 1. Inventaire des paires

In [ ]:
pairs = list_pairs(config.DATA_NOISY, config.DATA_CLEAN)
print(f'Nombre de paires alignées : {len(pairs)}')
if pairs:
    print('\n5 premiers fichiers :')
    for name, _, _ in pairs[:5]:
        print(f'  - {name}')

In [ ]:
# Fichiers présents dans un seul dossier (= paires manquantes)
from pathlib import Path
noisy_names = {p.name for p in Path(config.DATA_NOISY).glob('*') if p.is_file()}
clean_names = {p.name for p in Path(config.DATA_CLEAN).glob('*') if p.is_file()}
only_noisy = sorted(noisy_names - clean_names)
only_clean = sorted(clean_names - noisy_names)
print(f'Présents uniquement dans noisy/ : {len(only_noisy)}')
print(f'Présents uniquement dans clean/ : {len(only_clean)}')
if only_noisy[:5]:
    print('Ex noisy seul :', only_noisy[:5])
if only_clean[:5]:
    print('Ex clean seul :', only_clean[:5])

## 2. Distribution des durées et sample rates

In [ ]:
durations = []
sample_rates = Counter()
channels = Counter()

for name, noisy_path, _ in pairs:
    info = sf.info(noisy_path)
    durations.append(info.duration)
    sample_rates[info.samplerate] += 1
    channels[info.channels] += 1

durations = np.array(durations)
print(f'Durée  : min={durations.min():.2f}s | médiane={np.median(durations):.2f}s | '
      f'moy={durations.mean():.2f}s | max={durations.max():.2f}s')
print(f'Sample rates : {dict(sample_rates)}')
print(f'Canaux       : {dict(channels)}')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(durations, bins=30, color='steelblue', edgecolor='black')
ax.axvline(config.CLIP_DURATION, color='red', linestyle='--',
           label=f'CLIP_DURATION = {config.CLIP_DURATION}s')
ax.set_xlabel('Durée (s)')
ax.set_ylabel('Nombre de paires')
ax.set_title('Distribution des durées')
ax.legend()
plt.tight_layout()
plt.show()

pct_above = 100 * (durations >= config.CLIP_DURATION).mean()
print(f'{pct_above:.1f}% des clips font au moins {config.CLIP_DURATION}s (= pas de padding nécessaire).')

## 3. Aperçu d'une paire au hasard

In [ ]:
from IPython.display import Audio, display

idx = np.random.randint(0, len(pairs))
name, noisy_path, clean_path = pairs[idx]
print(f'Paire #{idx} : {name}')

noisy = A.load_audio(noisy_path)
clean = A.load_audio(clean_path)

print('Noisy :')
display(Audio(noisy, rate=config.SAMPLE_RATE))
print('Clean :')
display(Audio(clean, rate=config.SAMPLE_RATE))

A.plot_pair(noisy, clean)
plt.show()